A smaller version of the simulation (only one pqif, simulation and seed, and with small network), so I can monitor each variable, perform tests and get a quick log of what happens. One for oscillations and one for sequences.

This first cell is for code block tests

In [9]:
import numpy as np
amp_corriente = 20
N = 20


Ibac = amp_corriente*(2*np.random.uniform(size=N)-1)

print(Ibac.shape)

print(amp_corriente*(2*np.random.uniform(size=N)-1))

np.random.uniform
# print(np.random.uniform(size=N)-1)

(20,)
[-13.58288045 -18.13430326  -6.77823862  11.08064534 -15.32204318
  14.65404599   7.31268801 -11.0612993   -7.69823031  -2.06729057
   4.46949151  -9.9872312    7.16070919 -17.66609162   2.53419378
   4.0077116    6.33559994   2.48295282  11.23029435  -0.19280186]


<bound method RandomState.uniform of RandomState(MT19937) at 0x1BF2ECB7840>

# Oscillations

In [ ]:
# ========== Oscillations (parallelized) ==========

import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import pearsonr
import pandas as pd
import csv
import os
from joblib import Parallel, delayed

# Print log file:
import sys
log_fh = open('simulation_log_oscillations.txt',  mode='w', encoding='utf-8', buffering=1)
sys.stdout = log_fh

####### Global parameters #######
# Neurons
N = 20                 # Number of nodes (neurons)
N2 = int(N/2)           # Half

# Synaptic connections
p = 0.3                 # Probability of connection (non-zero elements in the weight matrix)
gsyn = 0.5              # Initial synaptic strength
alpha = 0.25            # Weight regularization parameter
# Dynamics
dt = 0.1                # Time step (time scale 10 ms)
itmax = 1000            # Number of iterations, where 1000 --> 1 sec
sigman = 1              # Noise standard deviation --> Noise in the dynamics
# Stimulus
itstim = 200            # Stimulation time
amp_corriente = 20      # Stimulus intensity
amp0 = 4                # Used in target. Changed from 8 to 4, in order to have the same current amplitudes as in the pre-training case for both oscillations and sequences
# Training
nloop = 16              # Number of loops, 0: pre-training, last: post-training
nloop_train = 10        # Last training loop
cant_seed = 50          # Independent simulations
ts = 5                  # 
b = 1 / ts              # adaptation parameter for r - In evolution of r, dr/dt = -b * r, b is magnitude
ftrain = 1              # Fraction of neurons to train
iout = np.arange(N)   # we monitor all neurons

# Containers for statistics
error_norms   = []          # ||error||_2 at each learning step
weight_norms  = []          # log‑norm ratio (same as modw in original)
motif_history = []          # (sigma2, tau_rec, …) per loop
dw_history = []

# History of variables
x_history = []
b_history = []
r_history = []
dx_history = []
dr_history = []
rls_history = []


# ------------------------------------------------------------
# One simulation configuration
# ------------------------------------------------------------
vt    = 0.0          # LIF spike‑threshold
vrest = -12.3        # LIF reset potential 

####### Function to generate target patterns #######

def generate_target(romega1, romega2, amp0):

    print("--------------------------------------------------\nGenerating target\n--------------------------------------------------")

    target=np.zeros((N,itmax))  # Initialize target (N, T)
    amp=np.random.uniform(size=N)*amp0  # Amplitude in [0, amp0)
    phase=np.random.uniform(0,2*np.pi,size=N)  # Phase in [0, 2pi)

    print(f"amp of shape {amp.shape}, phase of shape {phase.shape}")

    # Permuted indices for neurons
    indices = [i for i in range(N)]
    indices = np.random.permutation(indices) 
    print(f"Indices after permuting: {indices}")
    
    romega_vec = np.zeros(N)  # (N,)
    
    for i in range(N2):
        # Because it assigns based on permuted, it is randomized which half gets what indexed oscillations
        romega_vec[indices[i]]= romega1  
        romega_vec[indices[i+N2]]=romega2  

    print(f"Assigned theta and gamma to romega_vec")
    
    
    # Radians ?
    #    ``itmax`` is the total number of discrete time steps, therefore
    #    ``2π/itmax`` is the angular increment per step for a unit frequency.
    omega=romega_vec*2*np.pi/itmax 

    print(f"Converted to cycle-based frequencies")
    print(f"Filling the target matrix with oscillations in the form amp*cos(t*omega+phase) over time.")

    for it in range(itmax):
        target[:,it]=amp*np.cos(it*omega+phase) # Fill for all neurons per timestep. I.e. one iteration here is a vector of shape (N,) assigned to target in the it-th column

    # Show example
    import pandas as pd
    df_example = pd.DataFrame(target).round(4)
    print(f"Example part of target matrix of shape {target.shape}:")
    print(df_example.head(3).to_string())
            
    return target, amp, phase, omega, romega_vec, amp0



def dynamics(x_var,r_var,I_var,nqif, b):
    '''
    Computes dx/dt and dr/dt for the mixed QIF/LIF population

    Inputs:
        x_var   : internal state of neurons
        r_var   : output firing rate or adaptation variable
        I_var   : total input to neurons (external + recurrent)
        nqif    : number of QIF neurons at the start of the array
        b       : adaptation parameter for r
        
    Outputs:
        dx      : derivative of neuron state
        dr      : derivative of adaptation/firing rate
    '''
    # Initialize dx (derivative of state) as zeros for all neurons
    dx=np.zeros(N)
    # -----------------------------
    # Add stochastic noise to inputs
    I_noise_lif = np.random.randn(N - nqif)*sigman 
    I_noise_qif = np.random.randn(nqif)*sigman

    # -----------------------------
    # Compute derivative for LIF and QIF neurons

    # LIF dynamics: dx/dt = -x + I + noise
    dx[nqif:] = -x_var[nqif:] + I_var[nqif:] + I_noise_lif

    # QIF dynamics: dx/dt = 1 - cos(x) + I*(1 + cos(x)) + noise
    # This is a phase-based neuron model, x represents phase in [0, 2π)
    dx[:nqif] = 1 - np.cos(x_var[:nqif]) + I_var[:nqif]*(1 + np.cos(x_var[:nqif])) + I_noise_qif
    
    # -----------------------------
    # Compute derivative for adaptation variable r

    # Adaptation decays exponentially: dr/dt = -b * r
    dr = -b*r_var 

    dr_history.append(dr)
    dx_history.append(dx)
    b_history.append(b)

    return dx,dr


def detect(x,xnew,rnew,nspike,nqif, b, vt, vrest):

    # LIF spike detection
    ispike_lif=np.where(x[nqif:]<vt) and np.where(xnew[nqif:]>vt)
    ispike_lif=ispike_lif[0]+nqif
    if(len(ispike_lif)>0):
        rnew[ispike_lif[:]] = rnew[ispike_lif[:]] + b
        xnew[ispike_lif[:]] = vrest
        nspike[ispike_lif[:]] = nspike[ispike_lif[:]] + 1

    # QIF spike detection: Crossing pi
    dpi=np.mod(np.pi - np.mod(x,2*np.pi),2*np.pi)  # distance to pi
    ispike_qif=np.where((xnew[:nqif]-x[:nqif])>0) and np.where((xnew[:nqif]-x[:nqif]-dpi[:nqif])>0)
    if(len(ispike_qif)>0):
        rnew[ispike_qif[:]] = rnew[ispike_qif[:]] + b
        nspike[ispike_qif[:]] = nspike[ispike_qif[:]] + 1
    return xnew,rnew,nspike


def evolution(x, r, Iext, w, nqif, it, dt, iout, nspike, b, vt, vrest):
    '''
    One integration step + Spike detection
    
    Parameters 
    ----------------------------------------
    x : np.ndarray, shape (N,)
        neuron internal state vector  
    r : np.ndarray, shape (N,)
        neuron output                 
    Iext : np.ndarray, shape (N, itmax)
        external current from currents(). Column it contains stimulus at current time step (1 ms).
    w : np.ndarray, shape (N, N)
        Synaptic connectivity matrix, used to compute recurrent input wr
    nqif : int
        Number of QIF neurons. The first nqif entries of x and r is QIF
    it : int
        Current index, determines index of Iext
    iout : np.ndarray of ints
        Indices of the neurons whose output ('r') we want to record
    nspike : np.ndarray, shape (N)
        Counter that holds the number of spikes emitted in current iteration
    b : float
        Adaptation decay rate (1/ts). Also the increment added to r each time a spike occurs
    vt : float
        Threshold for LIF
    vrest : float
        Reset value for LIF

    Returns
    ----------------------------------------
    x, updated
    r, updated 
    nspike, spike count for this time step
    r[iout], output of neurons, is what is stored as outputs
    II (shape (N,)) it-th column of Iext
    v : Recurrent input w*r, to see contribution of networks own activity



    '''
    II = np.squeeze(np.asarray(Iext[:, it]))
    v = w.dot(r.T).A1
    dx, dr = dynamics(x, r, II + v, nqif, b)
    xnew = x + dt * dx / 2
    rnew = r + dt * dr / 2
    dx, dr = dynamics(xnew, rnew, II + v, nqif, b)
    xnew = x + dt * dx
    rnew = r + dt * dr
    xnew, rnew, nspike = detect(x, xnew, rnew, nspike, nqif, b, vt, vrest)
    x, r = np.copy(xnew), np.copy(rnew)

    x_history.append(x)
    r_history.append(r)

    return x, r, nspike, r[iout], II, v


def initialize_connectivity_matrix(N, p, gsyn):
    '''
       Create the recurrent weight matrix *W* 

    Parameters
    ----------
    N     : int
        Number of neurons in the network.
    p     : float, 0 ≤ p ≤ 1
        Probability that any given ordered pair (j → i) has a nonzero synaptic
        weight.  In other words, *p* controls the **sparseness** of the matrix.
    gsyn  : float
        Global scaling factor for the initial synaptic strengths.  After the
        random matrix is generated, each nonzero entry is multiplied by
        ``gsyn / sqrt(p*N)`` 

    Returns
    -------
    w : np.ndarray, shape (N, N)
        Dense weight matrix with the following properties:
        * No self connections (diagonal = 0).
        * Rowwise zero mean (each neuron receives a balanced set of
          excitatory/inhibitory inputs on average).
        * Overall variance set by ``gsyn`` and the connection probability ``p``

    '''
    w = sparse.random(N, N, p, data_rvs=np.random.randn).todense()
    np.fill_diagonal(w, 0)  # No autapses
    w *= gsyn / np.sqrt(p * N)  # rescale each weight so each entry: w_ij <- gsyn/sqrt(P*N) * w_ij
    # Zero mean for each row as in original
    for i in range(N):
        i0 = np.where(w[i, :])[1]  # find j neurons that project to i
        if len(i0) > 0:
            av0 = np.sum(w[i, i0]) / len(i0)
            w[i, i0] -= av0  # Subtract mean so each row has zero mean
    
    return w

def initialize_neurons(N):
    '''
    Initializes neurons

    Parameters
    --------------------
    N : Number of neurons

    Returns
    ----------
    x       : neuron internal state vector 
    r       : neuron output
    nspike  : initialized container for the spikes to N neurons

    '''
    x = np.random.uniform(size=N) * 2 * np.pi
    r = np.zeros(N)
    nspike = np.zeros(N)

    print(f"Initialized neuron internal state vector of shape {x.shape}, neuron output of shape {r.shape}, and nspike of shape {nspike.shape}")
    return x, r, nspike

def initialize_training(N, w):
    # Initialize correlation matrices for RLS learning
    # Creates the RLS structure (list of inverse-correlation matrices)
    nind=np.zeros(N).astype('int')  # (N,)
    idx=[] # list of presynaptic index vectors, one per neuron
    P=[]  # list of inverse-covariance matrices
    for i in range(N):
        ind=np.where(w[i,:])[1]  #Presynaptic neurons that project to i
        nind[i]=len(ind)  # in-degree of neuron i
        idx.append(ind)  # store list of presynaptic partners
        P.append(np.identity(nind[i])/alpha)   # list of correlation matrices; shape (len(ind), len(ind))

    print(f"Initialized training with P of type {type(P)}, length {len(P)}, with example matrices of shape {P[0].shape}, {P[2].shape}, {P[-1].shape}")

    print(f"Example P:")
    import pandas as pd
    df_example = pd.DataFrame(P[0]).round(4)
    print(df_example.to_string())

    return P, idx

def currents(N, itmax):
    ''' 
    Creates external current container (N, itmax=1000).

    Makes a baseline current for each neuron, where
    by drawing N numbers in [0,1), multiplying by 2 to 
    map them to [-1, 1]]. Then multiplying by amp_corriente 
    makes it [-amp_corriente, amp_corriente] per neuron.

    Then applies that baseline current only during stimulation
    window defined by itstim (first 200 steps)

    Parameters:
    ----------------------------------------
    N : Number of neurons
    itmax : number of iterations

    Returns
    ----------------------------------------
    Iext : input current (N neurons x T timesteps)

    '''
    Iext=np.zeros((N,itmax))  # (200, 1000)
    Ibac=amp_corriente*(2*np.random.uniform(size=N)-1)  # (N,)
    Iext[:, :itstim] = Ibac[:, None]  # Apply current during stimulation time
    print(f"Initialized input matrix of shape: {Iext.shape} with first {itstim} columns being injected external current of [-A, A]")
    return Iext  # The remaining 


def learning(it, iloop, w, r, P, idx, target, norm_w0):
    '''Recursive least squares weight update'''

    # error = f(t) - wr --> (N,) = (N,) - (N, N)*(N,) 
    error = target[:, it:it + 1] - w @ r.reshape(N, 1) # e(t) of shape (N,)

    neuron = 2

    for i in range(N):  # Per neuron
        ri = r[idx[i]].reshape(len(idx[i]), 1)  # (n_i, 1) column of presynaptic activities to neuron i
        k1 = P[i] @ ri  # P(t)r(t), shape (N_i, 1)
        k2 = ri.T @ P[i]  # r^T(t)P(t), shape (1, N_i)
        den = 1 + ri.T @ k1  # Denominator 1 + r^T(t)P(t)r(t), shape (1,1) (scalar)
        P[i] -= (k1 @ k2) / den  # New inverse corr. matrix
        dw = error[i, 0] * P[i] @ r[idx[i]]  # delta w = ePr
        w[i, idx[i]] += dw  # Apply delta w to appropriate columns
        # w now contains the new w, and P contains the inverse correlation matrices P(t+1)

        if i == neuron:
            # Example trace of what happens for one neuron
            rls_trace = (f"Timestep: {it} - Example history for neuron {i} in RLS\n--------------------\n  P: {P[i].shape}\n  ri: {ri.shape}\n  Computed error as e = f - wr, where\n  e: {error.shape}, f: {(target[:, it:it + 1]).shape}, w: {w.shape}, r: {(r.reshape(N, 1)).shape}\n  ri is shape: {ri.shape}\n  Pr becomes {k1.shape}\n  r^T * P becomes: {k2.shape}\n  Denominator 1 + r^T * P becomes: {den}\n  P updated becomes: {P[i].shape}\n  Delta w becomes: {dw.shape}\n  Updated w is {w.shape} and its row {(w[i, idx[i]]).shape} was updated with {dw.shape}\n--------------------")

    # if it % 10 == 0:
    #     modt_value = it + iloop * itmax
    #     modw = np.log(np.linalg.norm(w) / norm_w0)  # scalar


    modw = np.log(np.linalg.norm(w) / norm_w0)
    return w, P, error, modw, dw, rls_trace


####### Motifs and dimensionality calculations #######
            
def motifs(w,gsyn,N):
    w=w-np.mean(w)
    
    ww=np.matmul(w,w)
    wtw=np.matmul(w.T,w)
    wwt=np.matmul(w,w.T)
    
    sigma2=np.trace(wwt)/N
    
    tau_rec=np.trace(ww)
    tau_rec/=sigma2*N
    
    tau_div=np.sum(wwt)-np.trace(wwt)
    tau_div/=sigma2*N*(N-1)
    
    tau_con=np.sum(wtw)-np.trace(wtw)
    tau_con/=sigma2*N*(N-1)
    
    tau_chn=2*(np.sum(ww)-np.trace(ww))
    tau_chn/=sigma2*N*(N-1)
    
    return sigma2,tau_rec,tau_div,tau_con,tau_chn



# ------------------------------------------------------------

# One seed run

pqif = 0.5
nqif = int(N * pqif)

print(f"-------------------------------------------------- SIMULATION LOG EXAMPLE --------------------------------------------------\n")


# Initialize network
print(f"Initializing..")
x, r, nspike = initialize_neurons(N)
Iext = currents(N, itmax)  # Iext is the input matrix with

# for iext_it in Iext:
#     print(iext_it)

# print(f"Iext shape: {Iext.shape}")

# Initialize connectivity
w = initialize_connectivity_matrix(N, p, gsyn)
w_original = w  # A copy of the initialized connectivity matrix
norm_w0 = np.linalg.norm(w)

print("Initialized connectivity matrix:")
df_w = pd.DataFrame(w_original,
                    index=[f"N{i:02d}" for i in range(w_original.shape[0])],
                    columns=[f"N{j:02d}" for j in range(w_original.shape[1])])

# Show the raw numbers (rounded to 3 decimals for brevity)
print(df_w.round(2).head(2).to_string())




# Initialize RLS structures
P, idx = initialize_training(N, w)



# Generate target
target, phase, amp, omega, romega_vec, amp0 = generate_target(romega1=1, romega2=5, amp0=amp0)

print_counter = 0  # Some things I want to print examples in the loop, I use this to avoid printing each inner loop iteration

for iloop in range(nloop): # Epochs. nloop = 16
    # Each iteration is of itmax = 1000 millisecond = 1 second
    for it in range(itmax):  # itmax = 1000, it = 1 ms


        #  evolution
        x, r, nspike, rout, II, v = evolution(x, r, Iext, w, nqif, it, dt, iout, nspike, b, vt=vt, vrest=vrest)

        if print_counter == 0:
            print(f"Shapes of evolved variables:\n x: {x.shape}\n r: {r.shape}\n nspike: {nspike.shape}\n rout: {rout.shape}\n II: {II.shape}\n v: {v.shape}")
            print_counter += 1  # to avoid too many prints

        # learning - only after the stimulation window and only during training loops
        
        if iloop > 0 and iloop <= nloop_train and int(it > itstim):
            w, P, error, modw, dw, rls_trace = learning(it, iloop, w, r, P, idx, target, norm_w0)


            error_norms.append(np.linalg.norm(error))
            weight_norms.append(modw)
            dw_history.append(dw)
            rls_history.append(rls_trace)


    
    sigma2, tau_rec, tau_div, tau_con, tau_chn = motifs(w, gsyn, N)  # Compute motif statistics after loop


# -------------------------------------------------

#   PRINT STATISTICS

print("\n------ FINAL STATISTICS ------\n")

rls_history_chunk = rls_history[:2]

for rls in rls_history_chunk:
    print(rls)

# Shapes of various variables
print("------ Shapes / dimensions ------")
print(f"target              : {target.shape}")
print(f"error               : {error.shape}")
print(f"phase               : {phase.shape}")
print(f"amp                 : {amp.shape}")
print(f"omega               : {omega.shape}")
print(f"romega_vec          : {romega_vec.shape}")
print(f"connectivity matrix W: {w.shape}")
print(f"RLS inverse‑corr. matrices (list length) : {len(P)}")
print(f"  – first entry shape : {P[0].shape}  (typical incoming degree ≈ {P[0].shape[0]})")
print(f"presynaptic index list for neuron 0 : {idx[0].shape}")

# Error evolution
if error_norms:  # the errors appended
    print("\nLearning error (L2 norm) statistics")
    print(f"  # stored steps      : {len(error_norms)}")
    print(f"  min error          : {np.min(error_norms):.4e}")
    print(f"  max error          : {np.max(error_norms):.4e}")
    print(f"  final error        : {error_norms[-1]:.4e}")
else:
    print("\nNo learning step performed (training window may be empty).")

# Weight‑norm evolution
if weight_norms:  # modw
    print("\nWeight norm evolution (log ratio to initial norm)")
    print(f"  min log ratio : {np.min(weight_norms):.4f}")
    print(f"  max log ratio : {np.max(weight_norms):.4f}")
    print(f"  final log ratio : {weight_norms[-1]:.4f}")
else:
    print("\nNo weight updates performed (training loop was skipped).")

# Motif statistics per loop
print("\nMotif statistics per loop (sigma2, tau_rec, tau_div, tau_con, tau_chn)")
for loop, (sig2, tr, td, tc, tch) in enumerate(motif_history):
    print(f"  loop {loop:2d} → "
          f"sigma2={sig2:.4e}, "
          f"tau_rec={tr:.4e}, "
          f"tau_div={td:.4e}, "
          f"tau_con={tc:.4e}, "
          f"tau_chn={tch:.4e}")

# 5) Example of a single neuron’s RLS data
i = 0
print(f"\nNeuron {i} presynaptic pool size: {len(idx[i])}")
print(f"  Inverse‑corr. matrix shape : {P[i].shape}")
print(f"  Last error component       : {error[i,0]:.4e}")
print(f"  Last weight change (Δw)    : "
      f"{(w[i, idx[i]] - w[i, idx[i]]).shape}  (all zeros because we printed after the final update)")


# print(f"--- dx: \n {dx_history[0]}")
# print(f"--- dr: \n {dr_history[0]}")
# print(f"--- x:\n {x_history[0]}")
# print(f"--- r:\n {r_history[0]}")
# print(f"--- b history: {b_history}")

print("\n------ END OF RUN ------\n")
log_fh.close()
# sys.stdout = sys.__stdout__   # optional. restores the console
print(f"Simulation finished and stored in simulation_log_oscillations.txt")

'     N00  N01   N02   N03  N04   N05   N06  N07   N08  N09   N10   N11  N12   N13   N14   N15  N16   N17  N18  N19\nN00  0.0  0.0  0.23  0.06  0.0  0.00  0.01  0.0 -0.01  0.0  0.00  0.00  0.0  0.00 -0.09  0.00  0.0 -0.21  0.0  0.0\nN01  0.0  0.0  0.00  0.00  0.0  0.42  0.00  0.0  0.00  0.0 -0.04 -0.17  0.0 -0.39  0.00  0.29  0.0 -0.11  0.0  0.0'

'     0    1    2    3    4    5\n0  4.0  0.0  0.0  0.0  0.0  0.0\n1  0.0  4.0  0.0  0.0  0.0  0.0\n2  0.0  0.0  4.0  0.0  0.0  0.0\n3  0.0  0.0  0.0  4.0  0.0  0.0\n4  0.0  0.0  0.0  0.0  4.0  0.0\n5  0.0  0.0  0.0  0.0  0.0  4.0'

# Sequences